## Imports

In [ ]:
from datasets import load_dataset
from transformers import GPT2Tokenizer, GPT2LMHeadModel, Trainer, TrainingArguments, DataCollatorWithPadding
from transformers.trainer_utils import EvalPrediction
import torch
import pandas as pd
import evaluate  # For BLEU/ROUGE metrics
import numpy as np
import wandb  # For logging

In [ ]:
base_path = "/content/drive/MyDrive/pfe/gpt2"
data_path = 'https://raw.githubusercontent.com/abdelhaqelamraoui/pfe_product_description_generation/refs/heads/main/data/data_prepared.csv'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### A. Load and Clean Dataset

In [ ]:
# Load dataset
df = pd.read_csv(data_path)
# Remove duplicates and invalid entries
df = df.drop_duplicates().dropna()
# Filter out overly long inputs/outputs to reduce memory usage
df = df[df['input'].str.len() <= 200]  # Adjust threshold as needed
df = df[df['output'].str.len() <= 500]  # Adjust threshold as needed

dataset = Dataset.from_pandas(df)
# Stratified split (if applicable, e.g., by 'Style')
train_test_split = dataset.train_test_split(test_size=0.2, seed=42)
train_data = train_test_split['train']
test_data = train_test_split['test']

### B. Tokenize Data

In [ ]:
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token
# Add special token for input-output separation
tokenizer.add_special_tokens({'additional_special_tokens': ['<SEP>']})

def tokenize_function(examples):
    # Combine input and output with separator
    texts = [f"{inp}<SEP>{out}" for inp, out in zip(examples['input'], examples['output'])]
    encodings = tokenizer(
        texts,
        max_length=256,
        truncation=True,
        padding=False,  # Dynamic padding will be handled by DataCollator
        return_tensors='pt'
    )
    return {
        'input_ids': encodings['input_ids'],
        'attention_mask': encodings['attention_mask'],
        'labels': encodings['input_ids']
    }

# Tokenize datasets with larger batch size for efficiency
tokenized_train = train_data.map(tokenize_function, batched=True, batch_size=1000)
tokenized_test = test_data.map(tokenize_function, batched=True, batch_size=1000)

# Use DataCollator for dynamic padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

### C. Fine-Tune GPT-2

In [ ]:
# Initialize model
model = GPT2LMHeadModel.from_pretrained('gpt2')
model.resize_token_embeddings(len(tokenizer))  # Adjust for added special tokens

# Define compute_metrics for BLEU evaluation
bleu = evaluate.load('bleu')

def compute_metrics(eval_pred: EvalPrediction):
    predictions, labels = eval_pred
    # Decode predictions and labels
    predictions = tokenizer.batch_decode(np.argmax(predictions, axis=-1), skip_special_tokens=True)
    labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    # Extract output part after <SEP>
    predictions = [pred.split('<SEP>')[-1].strip() for pred in predictions]
    labels = [label.split('<SEP>')[-1].strip() for label in labels]
    results = bleu.compute(predictions=predictions, references=labels)
    return {'bleu': results['bleu']}

# Training arguments with optimizations
training_args = TrainingArguments(
    output_dir=f'{base_path}/gpt2-product-description',
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,  # Accumulate gradients for effective batch size of 8
    num_train_epochs=3,
    learning_rate=5e-5,  # Standard for GPT-2
    warmup_steps=500,  # Warmup for stable training
    lr_scheduler_type='cosine',  # Cosine annealing for better convergence
    fp16=True,  # Mixed precision training
    save_steps=500,  # Save more frequently for early stopping
    save_total_limit=2,
    logging_dir='./logs',
    logging_steps=100,
    eval_strategy='steps',
    eval_steps=500,
    load_best_model_at_end=True,  # Load best model based on validation loss
    metric_for_best_model='eval_loss',
    report_to='wandb',  # Log to Weights & Biases
)

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Start W&B run
wandb.init(project='gpt2-product-description')

# Train model
trainer.train()

### D. Generate Descriptions

In [ ]:
import torch

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

def generate_description(input_text, max_length=256, top_p=0.9, temperature=0.7):
    input_text = f'{input_text}<SEP>'
    inputs = tokenizer(input_text, return_tensors='pt', max_length=128, truncation=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    # Generate with top-p sampling
    outputs = model.generate(
        input_ids=inputs['input_ids'],
        attention_mask=inputs['attention_mask'],
        max_length=max_length,
        top_p=top_p,
        temperature=temperature,
        no_repeat_ngram_size=2,
        do_sample=True,  # Enable sampling
        early_stopping=True
    )
    
    description = tokenizer.decode(outputs[0], skip_special_tokens=True)
    description = description.split('<SEP>')[-1].strip().capitalize()
    return description

# Example
input_text = 'Style: Industrial | Material: Metal | Color: Gray | Dimensions: 30L x 20W x 15H'
print(generate_description(input_text))

### E. Save Model

In [ ]:
# Save model
model.save_pretrained(f'{base_path}/fine-tuned-gpt2')
tokenizer.save_pretrained(f'{base_path}/fine-tuned-gpt2')

# Load later
model = GPT2LMHeadModel.from_pretrained(f'{base_path}/fine-tuned-gpt2')
tokenizer = GPT2Tokenizer.from_pretrained(f'{base_path}/fine-tuned-gpt2')
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
# Load fine-tuned model and tokenizer
model = GPT2LMHeadModel.from_pretrained(f'{base_path}/fine-tuned-gpt2')
tokenizer = GPT2Tokenizer.from_pretrained(f'{base_path}/fine-tuned-gpt2')
tokenizer.pad_token = tokenizer.eos_token

def generate_product_description(input_features, max_length=256, top_p=0.9, temperature=0.7):
    """
    Generate product description from input features

    Args:
        input_features (str): Product features in format 'Style:...|Material:...|Color:...'
        max_length (int): Maximum length of generated description
        top_p (float): Top-p sampling parameter
        temperature (float): Temperature for sampling

    Returns:
        str: Generated product description
    """
    input_text = f'{input_features}<SEP>'
    inputs = tokenizer(
        input_text,
        max_length=128,
        truncation=True,
        return_tensors='pt'
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
            max_length=max_length,
            top_p=top_p,
            temperature=temperature,
            no_repeat_ngram_size=2,
            do_sample=True,
            early_stopping=True
        )

    description = tokenizer.decode(outputs[0], skip_special_tokens=True)
    description = description.split('<SEP>')[-1].strip().capitalize()

    return description

# Example usage
input_features = 'Style: Industrial | Material: Metal | Color: Gray | Dimensions: 30L x 20W x 15H | Features: Rust-proof, Wall-mounted'
generated_description = generate_product_description(input_features)

print('Input Features:')
print(input_features)
print('\nGenerated Description:')
print(generated_description)